# Algoritmos de optimización. Seminario

**Nombre y Apellidos:** Carlos y Irune

**Url:** https://github.com/jzalaya/grupo-algoritmos

**Problema:**

> 1. Sesiones de doblaje
> 2. Organizar los horarios de partidos de La Liga
> **3. Combinar cifras y operaciones**


## Descripción del problema

Disponemos de las nueve cifras del 1 al 9 (se excluye el cero) y de los cuatro signos de las
operaciones fundamentales: suma (+), resta (-), multiplicación (\*) y división (/).

Hay que combinarlos de forma alternada, una cifra y un operador, sin repetir ninguno de ellos,
para obtener una cantidad dada. Toda expresión válida tiene por tanto la forma

$$c_1 \; op_1 \; c_2 \; op_2 \; c_3 \; op_3 \; c_4 \; op_4 \; c_5$$

con cinco cifras distintas tomadas de $\{1,\dots,9\}$ y los cuatro operadores usados exactamente
una vez cada uno. El enunciado da como ejemplo, para obtener el 4:

$$4+2-6/3*1 = 4$$

Además de resolver el problema, el enunciado plantea dos preguntas de análisis:

- ¿Qué valor máximo y mínimo se pueden obtener?
- ¿Es posible encontrar todos los valores enteros entre dicho mínimo y dicho máximo?

### Hipótesis de modelado

El enunciado deja tres puntos abiertos que conviene fijar antes de programar nada, porque
cambian el conjunto de soluciones:

1. **Precedencia de operadores.** Se aplica la precedencia habitual: primero `*` y `/`, después
   `+` y `-`, evaluando de izquierda a derecha dentro de cada nivel. El ejemplo del enunciado lo
   confirma: `6/3*1` vale 2 y `4+2-2` da 4. Con evaluación estrictamente de izquierda a derecha
   esa misma expresión valdría `((4+2)-6)/3*1 = 0`, así que la precedencia estándar es la lectura
   correcta.
2. **Sin paréntesis.** No se contempla agrupar términos, la expresión es una cadena plana.
3. **División racional, no entera.** No se exige que cada división sea exacta, solo que el valor
   final coincida con el objetivo. Es la lectura natural y además la más general: si se exigiera
   divisibilidad en cada paso, el conjunto de valores alcanzables sería un subconjunto del que
   calculamos aquí. Los resultados intermedios se manejan como fracciones exactas por el motivo
   que se explica en el apartado de estructura de datos.


## Utilidades comunes

Antes de responder a las preguntas se define la representación de una expresión y su función de
evaluación, que se usan en todo el resto del notebook.

In [1]:
from fractions import Fraction
from itertools import permutations, product
import math
import random
import time

CIFRAS = tuple(range(1, 10))   # las nueve cifras disponibles, el cero queda excluido
OPERADORES = "+-*/"            # los cuatro operadores, cada uno se usa una sola vez
N_CIFRAS = 5                   # 5 cifras y 4 operadores intercalados


def evaluar(cifras, operadores):
    """Evalúa c0 op0 c1 op1 ... con la precedencia habitual (* y / antes que + y -).

    Recorre la expresión una sola vez manteniendo abierto el término multiplicativo en
    construcción y cerrándolo cuando aparece un + o un -. Trabaja con Fraction para que el
    resultado sea exacto y la comparación con el objetivo no dependa del error de redondeo.
    """
    total = Fraction(0)              # suma de los términos aditivos ya cerrados
    signo = 1                        # signo del término que se está construyendo
    termino = Fraction(cifras[0])    # valor parcial de ese término
    for op, c in zip(operadores, cifras[1:]):
        if op == '*':
            termino *= c
        elif op == '/':
            termino /= c
        else:                        # + o -: se cierra el término actual y se abre el siguiente
            total += signo * termino
            signo = 1 if op == '+' else -1
            termino = Fraction(c)
    return total + signo * termino


def texto(cifras, operadores):
    """Devuelve la expresión como cadena legible, solo para mostrar resultados."""
    partes = [str(cifras[0])]
    for op, c in zip(operadores, cifras[1:]):
        partes += [op, str(c)]
    return "".join(partes)


def formatea(valor):
    """Presenta un Fraction como entero si lo es, y si no como fracción y decimal."""
    if valor.denominator == 1:
        return str(valor.numerator)
    return f"{valor} ({float(valor):.4f})"


# Comprobación con el ejemplo del enunciado
_ej = ((4, 2, 6, 3, 1), ('+', '-', '/', '*'))
print(f"{texto(*_ej)} = {evaluar(*_ej)}")


4+2-6/3*1 = 4


## (\*) ¿Cuántas posibilidades hay sin tener en cuenta las restricciones?

## ¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?

### Respuesta

Conviene separar las dos restricciones que impone el enunciado, porque son independientes:
que las cinco cifras sean distintas y que los cuatro operadores sean distintos.

**Sin restricciones.** Si se permite repetir, cada uno de los 5 huecos de cifra admite 9 valores
y cada uno de los 4 huecos de operador admite 4:

$$9^5 \cdot 4^4 = 59\,049 \cdot 256 = 15\,116\,544$$

**Con restricciones.** Las cifras forman una variación sin repetición de 9 elementos tomados de
5 en 5, y los operadores una permutación de los 4:

$$V_{9,5} \cdot 4! = \frac{9!}{4!} \cdot 4! = 9! = 362\,880$$

La simplificación no es casualidad: elegir 5 cifras ordenadas de 9 y ordenar los 4 operadores
equivale a ordenar los 9 símbolos originales, de ahí que el resultado sea exactamente $9!$. Las
restricciones recortan el espacio en un factor de unas 42 veces, hasta un tamaño que sí permite
enumeración exhaustiva.

Un tercer número, que no pide el enunciado pero resulta decisivo para el diseño del algoritmo:
esas 362.880 expresiones producen solo **3.408 valores distintos**. Hay mucha redundancia
(por ejemplo `1+2` y `2+1` son expresiones diferentes con el mismo valor), y eso explica por qué
merece la pena resolver una vez para todos los objetivos en lugar de repetir la búsqueda.

In [2]:
# Fórmulas
sin_restricciones = 9 ** N_CIFRAS * 4 ** (N_CIFRAS - 1)
con_restricciones = math.perm(9, N_CIFRAS) * math.factorial(4)

print(f"Sin restricciones : 9^5 * 4^4       = {sin_restricciones:,}")
print(f"Con restricciones : V(9,5) * 4!     = {math.perm(9, N_CIFRAS):,} * 24 = {con_restricciones:,}")
print(f"Reducción         : x{sin_restricciones / con_restricciones:.1f}")
print(f"¿Coincide con 9!? : {con_restricciones == math.factorial(9)}")


Sin restricciones : 9^5 * 4^4       = 15,116,544
Con restricciones : V(9,5) * 4!     = 15,120 * 24 = 362,880
Reducción         : x41.7
¿Coincide con 9!? : True


In [3]:
# Validación de las dos fórmulas sobre una instancia reducida que sí se puede enumerar
# entera: 3 cifras tomadas de {1,2,3,4} y 2 operadores tomados de {+,-}.
pool_mini, ops_mini, k = (1, 2, 3, 4), "+-", 3

enum_sin_rep = sum(1 for _ in product(permutations(pool_mini, k), permutations(ops_mini, k - 1)))
enum_con_rep = sum(1 for _ in product(product(pool_mini, repeat=k), product(ops_mini, repeat=k - 1)))

print(f"sin repetición : enumerado {enum_sin_rep:>4}  fórmula {math.perm(4, k) * math.factorial(2):>4}")
print(f"con repetición : enumerado {enum_con_rep:>4}  fórmula {4 ** k * 2 ** (k - 1):>4}")


sin repetición : enumerado   48  fórmula   48
con repetición : enumerado  256  fórmula  256


## Modelo para el espacio de soluciones

## (\*) ¿Cuál es la estructura de datos que mejor se adapta al problema? Argumentalo. (Es posible que hayas elegido una al principio y veas la necesidad de cambiar, argumentalo)

### Respuesta

Una solución es una **secuencia ordenada**, no un conjunto: `4+2-6/3*1` y `2+4-6/3*1` usan las
mismas cifras y valen cosas distintas. Cualquier estructura que pierda el orden es inservible.

La representación elegida es un par de tuplas:

```python
cifras     = (4, 2, 6, 3, 1)      # tupla de 5 enteros distintos
operadores = ('+', '-', '/', '*') # tupla con los 4 operadores, uno de cada
```

Los motivos concretos:

- **Separar cifras y operadores** en lugar de una sola secuencia mezclada permite generarlas con
  dos `permutations` independientes, que es exactamente la estructura combinatoria del problema
  ($V_{9,5}$ por $4!$). Con una única lista intercalada habría que filtrar las combinaciones no
  válidas, desperdiciando trabajo.
- **Tupla y no lista** porque es inmutable y por tanto hashable. El análisis global necesita un
  `dict` que asocie cada valor alcanzable con una expresión que lo produce, y las claves de un
  diccionario tienen que ser hashables. Con listas habría que convertir en cada inserción.
- **`Fraction` y no `float`** para el valor. El objetivo se compara con `==`, y en coma flotante
  esa comparación es frágil por construcción.

**Cambio de enfoque durante el desarrollo.** El primer prototipo seguía la sugerencia del
enunciado: construir la expresión como cadena y evaluarla con `eval`. Es la versión más corta de
escribir, pero se descartó por tres razones que se midieron:

1. `eval` reanaliza la cadena entera en cada llamada. Medido sobre 3.000 expresiones, `eval` tarda
   del orden de 34 ms frente a menos de 3 ms de un evaluador propio en coma flotante, un factor de
   unas 12 veces.
2. `eval` devuelve `float`, con el problema de comparación ya mencionado.
3. Sobre todo, la cadena impide la **evaluación incremental**. El algoritmo mejorado necesita
   conocer el valor parcial de una expresión a medio construir para poder podar. Con una cadena
   habría que reevaluar desde cero en cada nodo del árbol de búsqueda, que es justo el coste que se
   quiere evitar.

El punto 1 tiene un matiz honesto: el evaluador propio con `Fraction` tarda en torno a 30 ms, es
decir, prácticamente lo mismo que `eval`. La aritmética exacta consume la ventaja del parseo. La celda
siguiente comprueba que, para este tamaño de problema, coma flotante y aritmética exacta nunca
discrepan al decidir si un valor es entero, así que la versión rápida sería utilizable; se mantiene
`Fraction` porque el coste absoluto es asumible y evita tener que razonar sobre tolerancias.

In [4]:
def evaluar_float(cifras, operadores):
    """Misma lógica que evaluar(), en coma flotante. Solo se usa para la comparación."""
    total, signo, termino = 0.0, 1.0, float(cifras[0])
    for op, c in zip(operadores, cifras[1:]):
        if op == '*':
            termino *= c
        elif op == '/':
            termino /= c
        else:
            total += signo * termino
            signo = 1.0 if op == '+' else -1.0
            termino = float(c)
    return total + signo * termino


# Coste de las tres alternativas sobre la misma muestra de expresiones. En el caso de
# eval se incluye la construcción de la cadena, porque es trabajo que ese enfoque exige.
from itertools import islice
muestra = [(c, ('+', '-', '*', '/')) for c in islice(permutations(CIFRAS, N_CIFRAS), 3000)]

for etiqueta, funcion in [("eval(cadena)", None),
                          ("propio float", evaluar_float),
                          ("propio Fraction", evaluar)]:
    t0 = time.perf_counter()
    if funcion is None:
        for c, o in muestra:
            eval(texto(c, o))
    else:
        for c, o in muestra:
            funcion(c, o)
    print(f"{etiqueta:<16} {1000 * (time.perf_counter() - t0):6.1f} ms")


eval(cadena)       31.7 ms
propio float        2.3 ms
propio Fraction    30.1 ms


In [5]:
# ¿Cambia alguna decisión al usar float en lugar de aritmética exacta?
# Se recorre el espacio completo comparando si ambas versiones coinciden al declarar
# que un valor es entero. Este es el único predicado del que dependen las respuestas.
discrepancias = 0
for c in permutations(CIFRAS, N_CIFRAS):
    for o in permutations(OPERADORES):
        vf, ve = evaluar_float(c, o), evaluar(c, o)
        if (vf == int(vf)) != (ve.denominator == 1):
            discrepancias += 1

print(f"Expresiones comparadas : {math.perm(9, N_CIFRAS) * 24:,}")
print(f"Discrepancias float/exacto : {discrepancias}")


Expresiones comparadas : 362,880
Discrepancias float/exacto : 0


## Según el modelo para el espacio de soluciones

## (\*) ¿Cuál es la función objetivo?

## (\*) ¿Es un problema de maximización o minimización?

### Respuesta

> **Pendiente.** Asignado a Irune (función objetivo, análisis de valores, juego de datos y cierre).
> El reparto completo está en `README.md`.

## Diseña un algoritmo para resolver el problema por fuerza bruta

### Respuesta

El espacio es de 362.880 expresiones, así que la enumeración exhaustiva es viable y sirve además
como referencia de corrección para todo lo demás.

El algoritmo genera cada variación de 5 cifras y, para cada una, las 24 permutaciones de
operadores, evalúa y compara con el objetivo:

```
para cada variación de 5 cifras distintas de {1..9}:
    para cada permutación de los 4 operadores:
        si evaluar(cifras, operadores) == objetivo:
            devolver la expresión
devolver "no existe solución"
```

Se implementan dos variantes, porque responden a preguntas distintas:

- `fuerza_bruta(objetivo)` se detiene en la primera solución. Es la respuesta al problema tal como
  lo plantea el enunciado y sirve de referencia para medir el algoritmo mejorado.
- `enumerar_todo()` hace una pasada completa y construye un diccionario `valor -> expresión`. Con
  ella se responden de golpe las preguntas de análisis (máximo, mínimo, enteros alcanzables) y
  cualquier objetivo posterior se resuelve en tiempo constante. Es la estrategia sensata cuando hay
  que responder a muchos objetivos: una única pasada de 362.880 evaluaciones frente a una búsqueda
  por objetivo.

In [6]:
def fuerza_bruta(objetivo, cifras_disponibles=CIFRAS):
    """Recorre el espacio completo y devuelve la primera expresión que vale 'objetivo'.

    Devuelve (solucion, evaluaciones), con solucion = None si el objetivo no es alcanzable.
    El contador de evaluaciones permite comparar el trabajo real con el del algoritmo podado.
    """
    objetivo = Fraction(objetivo)
    evaluaciones = 0
    for cifras in permutations(cifras_disponibles, N_CIFRAS):
        for ops in permutations(OPERADORES):
            evaluaciones += 1
            if evaluar(cifras, ops) == objetivo:
                return (cifras, ops), evaluaciones
    return None, evaluaciones


def enumerar_todo(cifras_disponibles=CIFRAS):
    """Pasada completa: devuelve {valor: (cifras, operadores)} con la primera expresión
    encontrada para cada valor alcanzable."""
    tabla = {}
    for cifras in permutations(cifras_disponibles, N_CIFRAS):
        for ops in permutations(OPERADORES):
            valor = evaluar(cifras, ops)
            if valor not in tabla:
                tabla[valor] = (cifras, ops)
    return tabla


# El objetivo 4 del enunciado, resuelto por fuerza bruta
sol, evaluaciones = fuerza_bruta(4)
print(f"objetivo 4 -> {texto(*sol)} = {evaluar(*sol)}   ({evaluaciones:,} evaluaciones)")


objetivo 4 -> 1-2*3/6+4 = 4   (250 evaluaciones)


In [7]:
# Pasada completa sobre el espacio de soluciones. Se reutiliza en varios apartados.
t0 = time.perf_counter()
TODAS = enumerar_todo()
t_enumeracion = time.perf_counter() - t0

print(f"Expresiones evaluadas : {math.perm(9, N_CIFRAS) * 24:,}")
print(f"Valores distintos     : {len(TODAS):,}")
print(f"Tiempo                : {t_enumeracion:.2f} s")


Expresiones evaluadas : 362,880
Valores distintos     : 3,408
Tiempo                : 4.02 s


## Calcula la complejidad del algoritmo por fuerza bruta

### Respuesta

Con $n$ cifras disponibles y expresiones de $k$ cifras y $k-1$ operadores tomados de un conjunto de
$m$ operadores distintos, el número de expresiones es

$$V_{n,k} \cdot \frac{m!}{(m-k+1)!} = \frac{n!}{(n-k)!} \cdot \frac{m!}{(m-k+1)!}$$

y cada evaluación cuesta $O(k)$, ya que el evaluador recorre la expresión una sola vez. El coste
total es por tanto

$$O\!\left( \frac{n!}{(n-k)!} \cdot \frac{m!}{(m-k+1)!} \cdot k \right)$$

Para los valores del enunciado ($n=9$, $k=5$, $m=4$) queda $V_{9,5} \cdot 4! \cdot 5 = 9! \cdot 5$,
del orden de $1{,}8$ millones de operaciones elementales. En espacio el algoritmo es $O(k)$ si solo
busca una solución, y $O$(número de valores distintos) en la variante que tabula, que empíricamente
son 3.408 entradas.

Lo relevante es el crecimiento: fijado $k=5$, el término $V_{n,5}$ crece como $n^5$, y si se dejara
crecer también la longitud de la expresión el crecimiento pasaría a ser factorial. La enumeración
exhaustiva funciona aquí porque $9!$ es pequeño, no porque el algoritmo escale.

La medición siguiente comprueba las dos cosas: que el número de expresiones generadas coincide con
la fórmula, y que el tiempo por expresión se mantiene constante, de modo que el tiempo total sigue
la curva de $V_{n,5} \cdot 4!$.

In [8]:
# Coste de la enumeración al ampliar el conjunto de cifras disponibles de 5 a 9.
print(f"{'n':>2} {'expresiones':>12} {'fórmula':>12} {'tiempo (s)':>11} {'µs/expr':>9}")
medidas = []
for n in range(5, 10):
    pool = CIFRAS[:n]
    t0 = time.perf_counter()
    n_expr = 0
    for c in permutations(pool, N_CIFRAS):
        for o in permutations(OPERADORES):
            evaluar(c, o)
            n_expr += 1
    dt = time.perf_counter() - t0
    medidas.append((n, n_expr, dt))
    formula = math.perm(n, N_CIFRAS) * math.factorial(4)
    print(f"{n:>2} {n_expr:>12,} {formula:>12,} {dt:>11.3f} {1e6 * dt / n_expr:>9.1f}")


 n  expresiones      fórmula  tiempo (s)   µs/expr
 5        2,880        2,880       0.030      10.6
 6       17,280       17,280       0.172      10.0


 7       60,480       60,480       0.623      10.3


 8      161,280      161,280       1.624      10.1


 9      362,880      362,880       3.805      10.5


## (\*) Diseña un algoritmo que mejore la complejidad del algoritmo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

### Respuesta

La fuerza bruta construye cada expresión entera antes de mirar cuánto vale. La mejora consiste en
**construirla de izquierda a derecha y abandonar en cuanto se sabe que ninguna forma de completarla
puede alcanzar el objetivo**, es decir, un esquema de vuelta atrás con poda por cotas
(*branch and bound*).

**Idea clave: descomposición en términos aditivos.** Fijado el patrón de operadores, la expresión
queda partida en términos separados por los `+` y `-`, y cada término es un producto y cociente de
cifras consecutivas. Por ejemplo, con el patrón `+ - * /`:

$$c_1 + c_2 - c_3 \cdot c_4 / c_5 \;=\; \underbrace{c_1}_{T_1} + \underbrace{c_2}_{T_2} - \underbrace{\frac{c_3 c_4}{c_5}}_{T_3}$$

El valor final es una suma con signo de esos términos. Esta estructura depende solo del patrón de
operadores, que se conoce nada más elegirlo, y es lo que permite acotar.

**Cota.** En un nodo intermedio se conoce la suma de los términos ya cerrados y el valor parcial del
término abierto. Falta repartir las cifras que quedan en el bolsillo. Para cada término pendiente,
formado por $a$ factores en el numerador y $b$ en el denominador, su valor máximo se obtiene
poniendo las $a$ cifras mayores del bolsillo arriba y las $b$ menores abajo; el mínimo, al revés.
Sumando esos extremos con el signo de cada término se obtiene un intervalo $[lo, hi]$ que contiene
con seguridad cualquier valor final alcanzable desde ese nodo. Si el objetivo cae fuera, la rama se
descarta.

**Por qué la cota es válida.** El cálculo permite que cada término elija sus cifras del bolsillo
completo, ignorando que dos términos no pueden compartir una cifra. Es una **relajación**: el
conjunto de valores que admite es un superconjunto de los realmente alcanzables, de modo que el
intervalo $[lo, hi]$ nunca se queda corto y la poda no puede descartar una solución existente. A
cambio la cota es algo más floja que la exacta, lo que solo cuesta explorar algún nodo de más. El
apartado de verificación comprueba esta corrección contra la enumeración exhaustiva en todo el rango
de enteros.

**Por qué mejora en la práctica.** Los términos alcanzan valores grandes (hasta $9 \cdot 8 = 72$),
así que en cuanto se fijan una o dos cifras el intervalo alcanzable se estrecha mucho y la mayoría
de los patrones de operadores quedan eliminados sin llegar a completar ninguna expresión. La poda
actúa además en los niveles altos del árbol, donde cada corte elimina un subárbol entero.

In [9]:
def _producto(xs):
    r = 1
    for x in xs:
        r *= x
    return r


def _plan_restante(ops_pendientes):
    """Descompone el sufijo de operadores todavía sin aplicar.

    Devuelve:
      (a, b)  factores que aún se van a añadir al numerador y al denominador del
              término que está abierto en este momento
      terminos  lista de (signo, a, b) de los términos que todavía no se han abierto
    """
    a = b = 0
    i = 0
    while i < len(ops_pendientes) and ops_pendientes[i] in '*/':   # el término abierto sigue creciendo
        if ops_pendientes[i] == '*':
            a += 1
        else:
            b += 1
        i += 1
    terminos = []
    while i < len(ops_pendientes):                                 # cada + o - abre un término nuevo
        signo = 1 if ops_pendientes[i] == '+' else -1
        i += 1
        a2 = b2 = 0
        while i < len(ops_pendientes) and ops_pendientes[i] in '*/':
            if ops_pendientes[i] == '*':
                a2 += 1
            else:
                b2 += 1
            i += 1
        terminos.append((signo, 1 + a2, b2))                       # 1 + a2: la cifra que abre el término
    return (a, b), terminos


def _extremos(bolsillo, a, b):
    """Valores mínimo y máximo de prod(a cifras) / prod(b cifras) usando cifras
    distintas del bolsillo. El máximo se logra con las mayores arriba y las menores
    abajo, y el mínimo al revés."""
    p = sorted(bolsillo)
    max_num = _producto(p[-a:]) if a else 1
    min_num = _producto(p[:a]) if a else 1
    max_den = _producto(p[-b:]) if b else 1
    min_den = _producto(p[:b]) if b else 1
    return Fraction(min_num, max_den), Fraction(max_num, min_den)


def _cotas(cerrado, signo, num, den, ops_pendientes, bolsillo):
    """Intervalo [lo, hi] que puede tomar la expresión completa desde este estado parcial.

    Relajación: cada término elige sus cifras del bolsillo entero, sin exigir que sean
    disjuntas entre términos. El intervalo resultante contiene por tanto todos los valores
    realmente alcanzables, que es lo que hace segura la poda.
    """
    (a, b), terminos = _plan_restante(ops_pendientes)

    # aportación del término que está abierto ahora mismo
    mn, mx = _extremos(bolsillo, a, b)
    t_lo, t_hi = Fraction(num, den) * mn, Fraction(num, den) * mx
    if signo > 0:
        lo, hi = cerrado + t_lo, cerrado + t_hi
    else:                                       # con signo negativo se invierten los extremos
        lo, hi = cerrado - t_hi, cerrado - t_lo

    # aportación de los términos aún sin abrir
    for s, a2, b2 in terminos:
        mn2, mx2 = _extremos(bolsillo, a2, b2)
        if s > 0:
            lo, hi = lo + mn2, hi + mx2
        else:
            lo, hi = lo - mx2, hi - mn2
    return lo, hi


def resolver(objetivo, cifras_disponibles=CIFRAS):
    """Vuelta atrás con poda por cotas. Devuelve (solucion, nodos_explorados).

    Se prueba cada patrón de operadores por separado, porque es el patrón el que fija la
    descomposición en términos de la que dependen las cotas.
    """
    objetivo = Fraction(objetivo)
    nodos = 0

    def backtrack(i, cerrado, signo, num, den, bolsillo, elegidas, ops):
        """i: cifras ya colocadas. cerrado: suma de términos cerrados.
        signo/num/den: término abierto. Devuelve la solución o None."""
        nonlocal nodos
        nodos += 1

        if i == N_CIFRAS:                                   # expresión completa
            return (tuple(elegidas), ops) if cerrado + signo * Fraction(num, den) == objetivo else None

        lo, hi = _cotas(cerrado, signo, num, den, ops[i - 1:], bolsillo)
        if not (lo <= objetivo <= hi):                      # poda: el objetivo es inalcanzable aquí
            return None

        op = ops[i - 1]
        for k, c in enumerate(bolsillo):
            resto = bolsillo[:k] + bolsillo[k + 1:]
            if op == '*':
                estado = (cerrado, signo, num * c, den)
            elif op == '/':
                estado = (cerrado, signo, num, den * c)
            else:                                           # se cierra el término y se abre otro
                estado = (cerrado + signo * Fraction(num, den), 1 if op == '+' else -1, c, 1)
            hallada = backtrack(i + 1, *estado, resto, elegidas + [c], ops)
            if hallada is not None:
                return hallada
        return None

    for ops in permutations(OPERADORES):
        for k, c in enumerate(cifras_disponibles):
            bolsillo = cifras_disponibles[:k] + cifras_disponibles[k + 1:]
            hallada = backtrack(1, Fraction(0), 1, c, 1, bolsillo, [c], ops)
            if hallada is not None:
                return hallada, nodos
    return None, nodos


sol, nodos = resolver(4)
print(f"objetivo 4 -> {texto(*sol)} = {evaluar(*sol)}   ({nodos} nodos explorados)")


objetivo 4 -> 1+4-2*3/6 = 4   (8 nodos explorados)


In [10]:
# Verificación de corrección: sobre todo el rango de enteros, el algoritmo podado debe
# coincidir con la enumeración exhaustiva en si el objetivo es alcanzable o no. Y cuando
# devuelve una expresión, esa expresión debe valer realmente el objetivo.
fallos = 0
for objetivo in range(-90, 100):
    encontrada, _ = resolver(objetivo)
    alcanzable = Fraction(objetivo) in TODAS
    if (encontrada is not None) != alcanzable:
        print(f"  discrepancia en {objetivo}: podado={encontrada is not None}, exhaustivo={alcanzable}")
        fallos += 1
    elif encontrada is not None and evaluar(*encontrada) != objetivo:
        print(f"  expresión incorrecta para {objetivo}: {texto(*encontrada)}")
        fallos += 1

print(f"Objetivos comprobados : {len(range(-90, 100))}")
print(f"Discrepancias         : {fallos}")


Objetivos comprobados : 190
Discrepancias         : 0


## (\*) Calcula la complejidad del algoritmo

### Respuesta

**En el peor caso la complejidad no baja.** La poda descarta ramas, pero no garantiza descartar
ninguna: si el objetivo estuviera dentro del intervalo alcanzable en todos los nodos, el algoritmo
recorrería el árbol completo, con las mismas
$O\!\left(\frac{n!}{(n-k)!} \cdot \frac{m!}{(m-k+1)!}\right)$ hojas que la fuerza bruta. A eso hay
que sumarle el coste de calcular la cota, que es $O(k \log k)$ por nodo por la ordenación del
bolsillo. El espacio sigue siendo $O(k)$, la profundidad de la pila de recursión.

Conviene decirlo con claridad: **esta mejora no cambia el orden asintótico, cambia el comportamiento
real**, que es lo habitual en los esquemas de ramificación y poda. Su valor está en el factor
constante efectivo, y ese factor es de varios órdenes de magnitud.

**En el caso medio** las mediciones son contundentes. Sobre los 147 objetivos enteros alcanzables el
algoritmo explora del orden de un centenar de nodos, frente a las decenas o centenas de miles de
evaluaciones de la fuerza bruta. Los objetivos **no alcanzables** son el caso interesante, porque
obligan a la fuerza bruta a recorrer las 362.880 expresiones para poder afirmar que no hay solución,
mientras que la poda lo descarta con unos cientos de nodos: ahí la diferencia llega a tres órdenes
de magnitud.

La razón de fondo es que el árbol es muy ancho en la raíz y las cotas son informativas muy pronto.
Basta fijar dos o tres cifras para que el intervalo alcanzable se estreche lo suficiente como para
eliminar patrones de operadores enteros.

In [11]:
# Nodos y tiempo del algoritmo podado, separando objetivos alcanzables de los que no lo son.
def perfil(objetivos):
    nodos_tot, tiempo_tot, peor = 0, 0.0, 0
    for t in objetivos:
        t0 = time.perf_counter()
        _, n = resolver(t)
        tiempo_tot += time.perf_counter() - t0
        nodos_tot += n
        peor = max(peor, n)
    k = len(objetivos)
    return nodos_tot / k, peor, 1000 * tiempo_tot / k


enteros_alcanzables = [int(v) for v in TODAS if v.denominator == 1]
rango = range(min(enteros_alcanzables), max(enteros_alcanzables) + 1)
no_alcanzables = [t for t in range(min(rango) - 25, max(rango) + 25) if t not in rango]

print(f"{'conjunto':<34} {'casos':>6} {'nodos medios':>13} {'peor':>7} {'ms medios':>10}")
for etiqueta, conjunto in [("objetivos alcanzables", sorted(enteros_alcanzables)),
                           ("objetivos no alcanzables", no_alcanzables)]:
    medio, peor, ms = perfil(conjunto)
    print(f"{etiqueta:<34} {len(conjunto):>6} {medio:>13.0f} {peor:>7} {ms:>10.1f}")


conjunto                            casos  nodos medios    peor  ms medios
objetivos alcanzables                 147           114     795        0.8


objetivos no alcanzables               49           265    1456        3.2


## Análisis del problema: valores máximo y mínimo, y enteros alcanzables

El enunciado plantea dos preguntas adicionales sobre el conjunto de valores que se pueden construir.

### Respuesta

> **Pendiente.** Asignado a Irune (función objetivo, análisis de valores, juego de datos y cierre).
> El reparto completo está en `README.md`.

## Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

### Respuesta

> **Pendiente.** Asignado a Irune (función objetivo, análisis de valores, juego de datos y cierre).
> El reparto completo está en `README.md`.

## Aplica el algoritmo al juego de datos generado

### Respuesta

> **Pendiente.** Asignado a Irune (función objetivo, análisis de valores, juego de datos y cierre).
> El reparto completo está en `README.md`.

## Enumera las referencias que has utilizado (si ha sido necesario) para llevar a cabo el trabajo

### Respuesta

> **Pendiente.** Asignado a Irune (función objetivo, análisis de valores, juego de datos y cierre).
> El reparto completo está en `README.md`.

## Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

### Respuesta

> **Pendiente.** Asignado a Irune (función objetivo, análisis de valores, juego de datos y cierre).
> El reparto completo está en `README.md`.